In [3]:
import duckdb
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import os

In [4]:
WAREHOUSE_TEST = "./test_duckdb_data/gtftest10.duckdb"

con = duckdb.connect(WAREHOUSE_TEST)
con.execute("SET TimeZone='UTC';")

con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,delays_with_support_columns
4,feed_info
5,routes
6,shapes
7,stop_times
8,stops
9,trips


In [5]:
con.sql("DESCRIBE trips_updates").df()


,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,route_id,VARCHAR,YES,None,None,None
2,direction_id,DOUBLE,YES,None,None,None
3,stop_id,BIGINT,YES,None,None,None
4,stop_sequence,BIGINT,YES,None,None,None
5,arrival_time,DOUBLE,YES,None,None,None
6,departure_time,DOUBLE,YES,None,None,None
7,arrival_dt,TIMESTAMP_NS,YES,None,None,None
8,departure_dt,TIMESTAMP_NS,YES,None,None,None


In [6]:
con.sql("DESCRIBE vehicle_positions").df()


,column_name,column_type,null,key,default,extra
0,trip_id,VARCHAR,YES,None,None,None
1,route_id,VARCHAR,YES,None,None,None
2,stop_id,BIGINT,YES,None,None,None
3,latitude,DOUBLE,YES,None,None,None
4,longitude,DOUBLE,YES,None,None,None
5,bearing,DOUBLE,YES,None,None,None
6,speed,DOUBLE,YES,None,None,None
7,timestamp,BIGINT,YES,None,None,None
8,vehicle_id,VARCHAR,YES,None,None,None
9,timestamp_dt,TIMESTAMP_NS,YES,None,None,None


In [7]:
# donnees temmps reel
con.sql("""
CREATE OR REPLACE TEMP VIEW actual AS
SELECT
  tu.trip_id,
  tu.stop_sequence,
  CAST(tu.arrival_dt AS TIMESTAMP WITH TIME ZONE) AS arrival_dt_utc
FROM trips_updates tu
WHERE tu.arrival_dt IS NOT NULL;
""")

con.sql("SELECT * FROM actual LIMIT 10").df()

,trip_id,stop_sequence,arrival_dt_utc
0,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,1,2025-09-11 11:45:42+00:00
1,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,2,2025-09-11 11:45:55+00:00
2,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,3,2025-09-11 11:46:44+00:00
3,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,4,2025-09-11 11:47:48+00:00
4,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,5,2025-09-11 11:48:50+00:00
5,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,6,2025-09-11 11:50:14+00:00
6,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,7,2025-09-11 11:51:09+00:00
7,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,8,2025-09-11 11:51:46+00:00
8,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,9,2025-09-11 11:52:12+00:00
9,5673239-NVN1_R_98_NVN101_13:45-PROJET2024-NVen...,11,2025-09-11 11:53:45+00:00


In [8]:
# time theorique

con.sql("""
CREATE OR REPLACE TEMP VIEW sched AS
SELECT
  st.trip_id,
  CAST(st.stop_sequence AS BIGINT) AS stop_sequence,
  st.stop_id,
  st.arrival_time_sec
FROM stop_times st
WHERE st.arrival_time_sec IS NOT NULL;
""")

con.sql("SELECT * FROM sched LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_time_sec
0,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,0,21681,30600
1,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,1,21652,30900
2,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,2,21644,32100
3,3064029-C32_A_1_C3201_08:30-RESEAU2021-C32-Lun...,3,21398,33300
4,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,0,21398,62400
5,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,1,21644,63600
6,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,2,21652,64800
7,3064030-C32_R_2_C3202_17:20-RESEAU2021-C32-Lun...,3,21681,65100
8,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,0,21681,61200
9,3064033-C32_A_1_C3202_17:00-RESEAU2021-C32-Wee...,1,21652,61500


In [9]:
# join entre time theorique - donnees reeel

con.sql("""
CREATE OR REPLACE TEMP VIEW joined AS
SELECT
  a.trip_id,
  a.stop_sequence,
  s.stop_id,
  a.arrival_dt_utc,
  CAST(a.arrival_dt_utc AT TIME ZONE 'Europe/Paris' AS DATE) AS local_service_day,
  s.arrival_time_sec
FROM actual a
JOIN sched s
  ON a.trip_id = s.trip_id
 AND a.stop_sequence = s.stop_sequence;
""")

con.sql("SELECT * FROM joined LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,local_service_day,arrival_time_sec
0,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,0,21583,2025-09-11 10:31:26+00:00,2025-09-11,45060
1,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,1,21261,2025-09-11 10:31:26+00:00,2025-09-11,45120
2,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,2,21265,2025-09-11 10:32:05+00:00,2025-09-11,45120
3,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,3,21266,2025-09-11 10:32:31+00:00,2025-09-11,45120
4,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,4,21267,2025-09-11 10:32:58+00:00,2025-09-11,45180
5,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,5,21268,2025-09-11 10:33:25+00:00,2025-09-11,45240
6,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,6,21269,2025-09-11 10:34:01+00:00,2025-09-11,45240
7,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,7,21270,2025-09-11 10:34:23+00:00,2025-09-11,45240
8,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,8,21271,2025-09-11 10:34:56+00:00,2025-09-11,45300
9,5203700-68_R_99_6802_12:31-PROJET2024-68-Semai...,9,21272,2025-09-11 10:35:35+00:00,2025-09-11,45360


In [10]:
# retard 

con.sql("""
CREATE OR REPLACE TEMP VIEW delays_with_support_columns AS
SELECT
  trip_id,
  stop_sequence,
  stop_id,
  arrival_dt_utc,
  (
    (local_service_day + arrival_time_sec * INTERVAL '1 second')
    AT TIME ZONE 'Europe/Paris'
  ) AS scheduled_ts_utc,
  EXTRACT(
    EPOCH FROM (
      arrival_dt_utc - (
        (local_service_day + arrival_time_sec * INTERVAL '1 second')
        AT TIME ZONE 'Europe/Paris'
      )
    )
  ) / 60.0 AS delay_min
FROM joined;
""")

con.sql("SELECT * FROM delays_with_support_columns ORDER BY arrival_dt_utc DESC LIMIT 10").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,scheduled_ts_utc,delay_min
0,4542749-44_R_99_4401_14:18-RESEAU2023-44-Semai...,7,4145,2025-09-11 12:44:41+00:00,2025-09-11 12:25:00+00:00,19.683333
1,4542693-44_A_50_4401_14:10-RESEAU2023-44-Semai...,7,4135,2025-09-11 12:33:42+00:00,2025-09-11 12:18:00+00:00,15.700000
2,6218561-84_R_95_8403_13:56-PROJET2025-84-Semai...,33,74,2025-09-11 12:30:32+00:00,2025-09-11 12:30:00+00:00,0.533333
3,6077772-58_A_50_5802_14:20-PROJET2025-58-Semai...,10,5162,2025-09-11 12:30:32+00:00,2025-09-11 12:31:00+00:00,-0.466667
4,4669793-62_A_46_6202_14:00-RESEAU2023-62-Semai...,28,6099,2025-09-11 12:30:27+00:00,2025-09-11 12:30:00+00:00,0.450000
5,6416089-21_R_99_2104_14:02-SETP2025-21-Semaine-40,20,4006,2025-09-11 12:30:25+00:00,2025-09-11 12:30:00+00:00,0.416667
6,6368868-09_R_99_0904_13:55-PROJET2025-09-Semai...,19,4269,2025-09-11 12:30:09+00:00,2025-09-11 12:30:00+00:00,0.150000
7,6428320-60_A_50_6001_14:10-PROJET2025-60-Semai...,20,2546,2025-09-11 12:30:09+00:00,2025-09-11 12:30:00+00:00,0.150000
8,6355833-18_A_50_1802_13:59-SETP2025-18-Semaine-39,26,376,2025-09-11 12:30:06+00:00,2025-09-11 12:30:00+00:00,0.100000
9,6369772-42_A_5_4203_14:00-PROJET2025-42-Semain...,22,5001,2025-09-11 12:30:00+00:00,2025-09-11 12:30:00+00:00,0.000000


In [11]:
# positions vehicles

con.sql("""
CREATE OR REPLACE TEMP VIEW vp AS
SELECT
  trip_id,
  route_id,
  latitude,
  longitude,
  to_timestamp("timestamp") AS ts_utc
FROM vehicle_positions;
""")

con.sql("SELECT * FROM vp ORDER BY ts_utc DESC LIMIT 5").df()


,trip_id,route_id,latitude,longitude,ts_utc
0,5909227-NVN3_A_90_NVN301_12:15-PROJET2025-NVen...,NVN3,43.721607,7.122200,2025-09-11 10:32:20+00:00
1,6365928-20_A_50_2005_11:53-SETP2025-20-Semaine-42,20,43.687618,7.181676,2025-09-11 10:32:20+00:00
2,6327127-17_A_50_1707_12:07-SETP2025-17-L-Ma-J-...,17,43.692310,7.199951,2025-09-11 10:32:20+00:00
3,6218549-84_R_99_8403_12:26-PROJET2025-84-Semai...,84,43.712494,7.333667,2025-09-11 10:32:20+00:00
4,6252480-11_A_50_1101_12:24-SETP2025-11-Semaine-37,11,43.718468,7.263024,2025-09-11 10:32:20+00:00


In [12]:
# association de retards par bus

con.sql("""
CREATE OR REPLACE TEMP VIEW bus_with_delay AS
SELECT
  v.trip_id,
  v.route_id,
  v.latitude,
  v.longitude,
  v.ts_utc,
  pe.delay_min
FROM vp v
LEFT JOIN delays_with_support_columns pe
  ON v.trip_id = pe.trip_id;
""")

con.sql("SELECT * FROM bus_with_delay ORDER BY ts_utc DESC LIMIT 10").df()


,trip_id,route_id,latitude,longitude,ts_utc,delay_min
0,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.050000
1,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.566667
2,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.550000
3,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.433333
4,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.383333
5,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.800000
6,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,1.300000
7,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,0.850000
8,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,1.700000
9,6218532-84_A_46_8402_12:20-PROJET2025-84-Semai...,84,43.71693,7.315751,2025-09-11 10:32:20+00:00,-1.433333


In [13]:
EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

def current_timestamp_string():
    return datetime.now(ZoneInfo("Europe/Paris")).strftime("%Y%m%d_%H%M%S")

df = con.sql("""
SELECT
  trip_id,
  route_id,
  latitude,
  longitude,
  ts_utc AT TIME ZONE 'Europe/Paris' AS timestamp_paris,
  ROUND(delay_min, 2) AS delay_min
FROM bus_with_delay
ORDER BY ts_utc DESC;
""").df()

csv_path = f"{EXPORT_DIR}/position_bus_vehicles_{current_timestamp_string()}.csv"
df.to_csv(csv_path, index=False)
csv_path

'./exports/position_bus_vehicles_20250911_135639.csv'